# ДЗ №3. Ранжирование на основе datamart

Общая информация
Дата выдачи: 3 апреля 2026

Дедлайн: 26 апреля 2026 23:59 MSK

В этом домашнем задании мы продолжим строить приближенную к реальной рекомендательную систему. Работать будем с данными marketplace из [T-ECD](https://huggingface.co/datasets/t-tech/T-ECD).

Обычно рекомендательная система состоит из нескольких этапов:
1. Отбор кандидатов (Retrieval)
2. Ранжирование (Ranking)
3. Бизнес-логика (например, условие на то, чтобы товары от одного продавца не стояли в ленте друг за другом)

В этом домашнем задании сосредоточимся на втором этапе. Можно и нужно использовать наработки из предыдущего домашнего задания!

Краткое напоминание, почему отбор кандидатов и ранжирование - разные этапы. Задачу рекомендаций можно решать как регрессию (насколько релевантен айтем), классификацию (релевантен ли айтем) или ранжирование (какой из двух айтемов релевантнее). В идеале - проранжировать каталог под каждого пользователя. Но каталог всегда существенно больше того подмножества айтемов, которые пользователь увидит в итоговой выдаче. А качественно ранжировать весь каталог - ОЧЕНЬ долго и дорого. Получаем trade-off скорости и качества. Простое решение - многостадийные рекомендации. Сначала отберем кандидатов (релевантные/не релевантные), а потом проранжируем только релевантные.

В этом задании следующая разбалловка:

1) Cбор датамарта - 4 балла
2) Сбор датасета для обучения ранжирования - 2 балла
3) Сбор град. бустинга и оценка по метрикам с бейзлайном - 4 балла

Соответственно, максимум можно набрать 10 баллов.

In [1]:
!pip install -q polars lightgbm scikit-learn catboost shap optuna implicit torch

In [1]:
import json
import gc
import os
import random
import typing as t
from abc import ABC, abstractmethod
from collections import defaultdict
from dataclasses import dataclass
from functools import partial
from pathlib import Path

import joblib
import lightgbm as lgbm
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import polars as pl
import polars.selectors as cs
import shap
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import HTML
from implicit.als import AlternatingLeastSquares
from optuna.samplers import TPESampler
from scipy.sparse import coo_matrix, csr_matrix
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, Dataset, IterableDataset

## Download Data

Данные занимают около 3.5 GB

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="t-tech/T-ECD",
    repo_type="dataset",
    local_dir=".",
    local_dir_use_symlinks=False,
    allow_patterns=["dataset/small/users.pq", "dataset/small/marketplace/**"]
)

Больше всего места занимают эмбеддинги айтемов, их по желанию можно удалить, так как в этой работе нам потребуется дополнительное место на создание датамарта

In [68]:
pl.read_parquet("dataset/small/marketplace/items.pq").drop("embedding").write_parquet("dataset/small/marketplace/items.pq")

## I. Datamart (4 балла)

В задаче ранжирования хорошо себя показывают бустинги ([Catboost](https://catboost.ai), [LGBM](https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.Booster.html), [XGBoost](https://xgboost.readthedocs.io/en/stable/)). С точки зрения интерфейса: Algorithm(user, item, [features]), где фичи - любые полезные статистики (количество просмотров, конверсия из клика в кликаут, средний рейтинг, ...).

Каждый раз рассчитывать фичи с нуля по сырым логам достаточно тяжело (а в реальности рекомендации мы делаем не один раз в домашней работе, а гораздо чаще). Учитывая, что логи могут иметь разный формат или нуждаться в  дополнительной фильтрации (баги в логах всё же не редкость). Простое решение - предподсчитать статистики и сохранить их в отдельных файлах, которые затем удобно просто прочитать. 

Это удобно сделать посредством датамарта. Датамарт - это витрина данных под определенную задачу. Он состоит из слоёв, где переход между слоями задает преобразование над данными, и обычно такие преобразования выполняются раз в какое-то время (в нашем случае пусть будет день). Для задачи ранжирования нам потребуется три слоя: 
1. Raw - содержит сырые логи за каждый день. Мы уже собрали его на предыдущем шаге.
2. Aggs - содержит агрегированные статистики по юзерам и айтемам за каждый день (user, item, [stats]). Например, количество просмотров, количество кликов, количество кликов c поверхности поиска.
3. Features - содержит фичи, которые мы хотим использовать в модели, тоже за каждый день. Например, средний рейтинг айтема, средний рейтинг айтема по категориям, общая конверсия из клика в кликаут для пользователя за последние 30 дней. На этом слое удобно выделить отдельные папки по группам фичей (user, item, user-item, ...), чтобы избежать дубликатов при хранении. 

Возьмем только небольшой срез данных, иначе дальнейшая работа может стать computationally infeasible. Переложим этот срез в `datamart/raw/events/{action_type}/{day}.pq`

Именно в таком формате логи обычно хранятся в сервисе. 

Вы можете расширить условия на сэмплирование юзеров и айтемов. Если у вас будут проблемы с памятью, то можете и уменьшить что-то, но чем меньше ваш датасет, тем хуже будут метрики у конечной модели

In [88]:
ACTION_TYPES = ["view", "click", "clickout", "like"]
SUBDOMAINS = ["u2i", "i2i", "catalog", "search", "other"]

DAYS = list(range(1250, 1301))  # не все даты

Будем считать, что view < click < clickout < like с точки зрения бизнеса. Этот факт будет использоваться далее.

In [89]:
selected_users = (
    pl.concat(
        [pl.scan_parquet(f"dataset/small/marketplace/events/{str(day).zfill(5)}.pq") for day in DAYS[-10:]]
    )
    .group_by("user_id").agg(pl.len()).sort("len", descending=True)
    .head(20000).collect()["user_id"].to_list()
)

selected_items = (
    pl.concat(
        [pl.scan_parquet(f"dataset/small/marketplace/events/{str(day).zfill(5)}.pq") for day in DAYS[-10:]]
    ).group_by("item_id").agg(pl.len()).sort("len", descending=True)
    .head(20000).collect()["item_id"].to_list()
)

In [90]:
USERS = pl.scan_parquet("dataset/small/users.pq").filter(pl.col("user_id").is_in(selected_users)).collect()
print(USERS.shape)
ITEMS = pl.scan_parquet("dataset/small/marketplace/items.pq").filter(pl.col("item_id").is_in(selected_items)).collect()

(20000, 3)


### I.I. Datamart -> Raw слой (1 из 4 баллов)

Здесь вам надо собрать  raw слой:

![](images/datamart_raw.png)

в каждом файлике должны храниться данные на конкретную дату


In [ ]:
## YOUR CODE HERE

  0%|          | 0/51 [00:00<?, ?it/s]

### I.II. Datamart -> Agg слой (1 из 4 баллов)

Рассчитате количество событий каждого типа (`action_type`) по каждой поверхности (`subdomain`) по парам (`user_id`, `item_id`) за каждый день. Не забудьте про общий счетчик - сумму по всем поверхностям. Сохраните в виде polars-таблиц.

Аналогично raw, но в agg значения внутри дня должны быть агрегированы

![](images/datamart_agg.png)

In [ ]:
events_dir = Path("datamart/aggs/events/")
os.makedirs(events_dir, exist_ok=True)

for day in tqdm(DAYS):
    ## YOUR CODE HERE

  0%|          | 0/51 [00:00<?, ?it/s]

Пример того, что может получиться.

In [93]:
pl.read_parquet("datamart/aggs/events/01300.pq").sample(10)

user_id,item_id,num_view_all_subdomains,num_view_u2i,num_view_i2i,num_view_catalog,num_view_search,num_view_other,num_click_all_subdomains,num_click_u2i,num_click_i2i,num_click_catalog,num_click_search,num_click_other,num_clickout_all_subdomains,num_clickout_u2i,num_clickout_i2i,num_clickout_catalog,num_clickout_search,num_clickout_other,num_like_all_subdomains,num_like_u2i,num_like_i2i,num_like_catalog,num_like_search,num_like_other
u64,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
19096353,"""nfmcg_17913509""",1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
57052792,"""nfmcg_8480860""",1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
74876789,"""nfmcg_21984785""",2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
78717331,"""nfmcg_271934""",1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14532028,"""nfmcg_28240670""",1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10191967,"""nfmcg_8425753""",14.0,0.0,14.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
27365203,"""nfmcg_14205552""",1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31863641,"""nfmcg_18106422""",1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
84827754,"""nfmcg_22967103""",2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### I.III. Datamart -> Feature слой (2 из 4 баллов)

Подумайте, какие признаки, рассчитанные на основе ранее собранных статистик, можно будет использовать в качестве фичей для модели. Реализуйте логику их подсчета за каждый день. Обратите внимание, что все фичи необходимо рассчитывать по какому-то временному окну, например "количество кликов на товар за последние 14 дней".

Сохраните признаки в виде polars-таблиц. Не забудьте про декомпозицию на отдельные папки по сущностям с целью избежать дубликатов при хранении.

Минимально должно получиться 10 признаков, из которых:
* 2 должны относиться только к сущности "пользователь" (например, медианная цена кликнутых айтемов у этого пользователя)
* 2 должны относиться только к сущности "айтем" (например, средняя конверсия из клика в кликаут по поверхности "поиск" у этого айтема)
* 6 признаков, которые показывают связь пользователя и айтема (например, количество просмотров этого айтема у этого пользователя).

Однако настоятельно рекомендуется собрать больше признаков. В данных много сущностей - категория, соцдем-кластер, бренд. И много параметров - цена, поверхность, тип события. В том числе можно рассчитать "изменение признака относительно предыдущего дня". Используйте polars expressions для написания шаблонного кода, в который затем удобно подставить конкретные названия сущностей и параметров, и получить готовый набор признаков. 

P.S. Цену для товаров мы считаем фиксированной, она представлена в каталоге `ITEMS`

Может быть полезно посчитать конверсии как отношение числа второго события из пары к числу первого события из пары. Аккуратнее с делением на ноль - возможно, пользователей, для которых не определено число первого события из пары, не стоит учитывать при расчетах.

Пример того, как должно получиться:

![](images/datamart_feat2.png)

In [94]:
CONVERSION_PAIRS = [
    ("view", "click"), 
    ("view", "clickout"), 
    ("view", "like"),
    ("click", "clickout"),
    ("click", "like"),
    ("clickout", "like"),
]

Удобно называть фичи следующим образом.

In [95]:
def _feature_name(
    feature: str,
    keys: list[str],
    type: t.Literal["num", "cat"] = "num",
) -> str:
    return f"f_num__{'_'.join(keys)}__{feature}"

In [ ]:
def calculate_user_group_item_group_features(
    day_from: int = 1200,
    day_to: int = 1300,
    num_days: int = 14,
    user_group: t.Literal["region", "socdem_cluster"] | None = None,
    item_group: t.Literal["brand_id", "category", "subcategory", "item_id"] = "item_id"
) -> None:
    global USERS, ITEMS

    ## YOUR CODE HERE

def calculate_user_item_group_features(
    day_from: int = 1200,
    day_to: int = 1300,
    num_days: int = 14,
    item_group: t.Literal["item_id", "brand_id", "category", "subcategory"] | None = None
) -> None:
    
    ## YOUR CODE HERE

    


In [97]:
for item_group in [None, "item_id", "brand_id", "category"]:
    print(f"Calculating features for user_id and {item_group}")
    calculate_user_item_group_features(
        num_days=30, item_group=item_group
    )

Calculating features for user_id and None


  0%|          | 0/101 [00:00<?, ?it/s]

Calculating features for user_id and item_id


  0%|          | 0/101 [00:00<?, ?it/s]

Calculating features for user_id and brand_id


  0%|          | 0/101 [00:00<?, ?it/s]

Calculating features for user_id and category


  0%|          | 0/101 [00:00<?, ?it/s]

In [98]:
for user_group in [None, "socdem_cluster"]:
    for item_group in ["item_id", "brand_id", "category"]:
        print(f"Calculating features for {user_group} and {item_group}")
        calculate_user_group_item_group_features(
            num_days=30, user_group=user_group, item_group=item_group
        )

Calculating features for None and item_id


  0%|          | 0/101 [00:00<?, ?it/s]

Calculating features for None and brand_id


  0%|          | 0/101 [00:00<?, ?it/s]

Calculating features for None and category


...

  0%|          | 0/101 [00:00<?, ?it/s]

Calculating features for socdem_cluster and brand_id


  0%|          | 0/101 [00:00<?, ?it/s]

Calculating features for socdem_cluster and category


  0%|          | 0/101 [00:00<?, ?it/s]

Пример того, что может получиться.

In [99]:
!ls datamart/features/events

brand_id                socdem_cluster-category user_id-category
category                socdem_cluster-item_id  user_id-item_id
item_id                 user_id
socdem_cluster-brand_id user_id-brand_id


In [100]:
pl.read_parquet("datamart/features/events/user_id/01250.pq").sample(10)

user_id,f_num__user_id__num_view_30d,f_num__user_id__num_view_from_u2i_30d,f_num__user_id__num_view_from_i2i_30d,f_num__user_id__num_view_from_catalog_30d,f_num__user_id__num_view_from_search_30d,f_num__user_id__num_view_from_other_30d,f_num__user_id__num_click_30d,f_num__user_id__num_click_from_u2i_30d,f_num__user_id__num_click_from_i2i_30d,f_num__user_id__num_click_from_catalog_30d,f_num__user_id__num_click_from_search_30d,f_num__user_id__num_click_from_other_30d,f_num__user_id__num_clickout_30d,f_num__user_id__num_clickout_from_u2i_30d,f_num__user_id__num_clickout_from_i2i_30d,f_num__user_id__num_clickout_from_catalog_30d,f_num__user_id__num_clickout_from_search_30d,f_num__user_id__num_clickout_from_other_30d,f_num__user_id__num_like_30d,f_num__user_id__num_like_from_u2i_30d,f_num__user_id__num_like_from_i2i_30d,f_num__user_id__num_like_from_catalog_30d,f_num__user_id__num_like_from_search_30d,f_num__user_id__num_like_from_other_30d,f_num__user_id__median_price_view_30d,f_num__user_id__median_price_view_from_u2i_30d,f_num__user_id__median_price_view_from_i2i_30d,f_num__user_id__median_price_view_from_catalog_30d,f_num__user_id__median_price_view_from_search_30d,f_num__user_id__median_price_view_from_other_30d,f_num__user_id__median_price_click_30d,f_num__user_id__median_price_click_from_u2i_30d,f_num__user_id__median_price_click_from_i2i_30d,f_num__user_id__median_price_click_from_catalog_30d,f_num__user_id__median_price_click_from_search_30d,f_num__user_id__median_price_click_from_other_30d,f_num__user_id__median_price_clickout_30d,f_num__user_id__median_price_clickout_from_u2i_30d,f_num__user_id__median_price_clickout_from_i2i_30d,f_num__user_id__median_price_clickout_from_catalog_30d,f_num__user_id__median_price_clickout_from_search_30d,f_num__user_id__median_price_clickout_from_other_30d,f_num__user_id__median_price_like_30d,f_num__user_id__median_price_like_from_u2i_30d,f_num__user_id__median_price_like_from_i2i_30d,f_num__user_id__median_price_like_from_catalog_30d,f_num__user_id__median_price_like_from_search_30d,f_num__user_id__median_price_like_from_other_30d
u64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
28095731,23.0,11.0,0.0,12.0,0.0,0.0,2.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.242515,2.242515,null,null,null,null,5.386636,null,null,5.386636,null,null,null,null,null,null,null,null,null,null,null,null,null,null
8204586,26.0,6.0,2.0,9.0,0.0,9.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.211659,-0.231156,3.54582,-0.225814,null,1.644707,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
60979916,49.0,11.0,0.0,12.0,24.0,2.0,4.0,1.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.722465,1.128698,null,0.589691,3.789058,4.100957,4.245498,4.359318,null,null,4.131678,null,null,null,null,null,null,null,null,null,null,null,null,null
44601266,4.0,3.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.461099,1.174263,null,null,null,1.747936,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
9083934,3.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.138842,null,null,null,-3.138842,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
37496392,4.0,3.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.370631,3.370631,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
25838143,11.0,3.0,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.600727,-0.602695,null,null,null,2.885351,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
30

In [101]:
pl.read_parquet("datamart/features/events/item_id/01250.pq").sample(10)

item_id,f_num__item_id__conv_from_view_to_click_30d,f_num__item_id__conv_from_view_to_click_from_u2i_30d,f_num__item_id__conv_from_view_to_click_from_i2i_30d,f_num__item_id__conv_from_view_to_click_from_catalog_30d,f_num__item_id__conv_from_view_to_click_from_search_30d,f_num__item_id__conv_from_view_to_click_from_other_30d,f_num__item_id__conv_from_view_to_clickout_30d,f_num__item_id__conv_from_view_to_clickout_from_u2i_30d,f_num__item_id__conv_from_view_to_clickout_from_i2i_30d,f_num__item_id__conv_from_view_to_clickout_from_catalog_30d,f_num__item_id__conv_from_view_to_clickout_from_search_30d,f_num__item_id__conv_from_view_to_clickout_from_other_30d,f_num__item_id__conv_from_view_to_like_30d,f_num__item_id__conv_from_view_to_like_from_u2i_30d,f_num__item_id__conv_from_view_to_like_from_i2i_30d,f_num__item_id__conv_from_view_to_like_from_catalog_30d,f_num__item_id__conv_from_view_to_like_from_search_30d,f_num__item_id__conv_from_view_to_like_from_other_30d,f_num__item_id__conv_from_click_to_clickout_30d,f_num__item_id__conv_from_click_to_clickout_from_u2i_30d,f_num__item_id__conv_from_click_to_clickout_from_i2i_30d,f_num__item_id__conv_from_click_to_clickout_from_catalog_30d,f_num__item_id__conv_from_click_to_clickout_from_search_30d,f_num__item_id__conv_from_click_to_clickout_from_other_30d,f_num__item_id__conv_from_click_to_like_30d,f_num__item_id__conv_from_click_to_like_from_u2i_30d,f_num__item_id__conv_from_click_to_like_from_i2i_30d,f_num__item_id__conv_from_click_to_like_from_catalog_30d,f_num__item_id__conv_from_click_to_like_from_search_30d,f_num__item_id__conv_from_click_to_like_from_other_30d,f_num__item_id__conv_from_clickout_to_like_30d,f_num__item_id__conv_from_clickout_to_like_from_u2i_30d,f_num__item_id__conv_from_clickout_to_like_from_i2i_30d,f_num__item_id__conv_from_clickout_to_like_from_catalog_30d,f_num__item_id__conv_from_clickout_to_like_from_search_30d,f_num__item_id__conv_from_clickout_to_like_from_other_30d,f_num__item_id__share_view_30d,f_num__item_id__share_view_from_u2i_30d,f_num__item_id__share_view_from_i2i_30d,f_num__item_id__share_view_from_catalog_30d,f_num__item_id__share_view_from_search_30d,f_num__item_id__share_view_from_other_30d,f_num__item_id__share_click_30d,f_num__item_id__share_click_from_u2i_30d,f_num__item_id__share_click_from_i2i_30d,f_num__item_id__share_click_from_catalog_30d,f_num__item_id__share_click_from_search_30d,f_num__item_id__share_click_from_other_30d,f_num__item_id__share_clickout_30d,f_num__item_id__share_clickout_from_u2i_30d,f_num__item_id__share_clickout_from_i2i_30d,f_num__item_id__share_clickout_from_catalog_30d,f_num__item_id__share_clickout_from_search_30d,f_num__item_id__share_clickout_from_other_30d,f_num__item_id__share_like_30d,f_num__item_id__share_like_from_u2i_30d,f_num__item_id__share_like_from_i2i_30d,f_num__item_id__share_like_from_catalog_30d,f_num__item_id__share_like_from_search_30d,f_num__item_id__share_like_from_other_30d
str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""nfmcg_19368599""",0.0,null,0.0,0.0,0.0,0.0,0.0,null,0.0,0.0,0.0,0.0,0.02,null,0.0,0.0,0.0,1.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.001021,0.0,0.000388,0.000245,0.000408,0.00002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002392,0.0,0.0,0.0,0.0,0.002392
"""nfmcg_12291960""",0.038961,0.05,0.028986,0.0,0.117647,0.0,0.006494,0.025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.166667,0.5,0.0,null,0.0,null,0.0,0.0,0.0,null,0.0,null,0.0,0.0,null,null,null,null,0.003144,0.000817,0.001409,0.00051,0.000347,0.000184,0.000305,0.000102,0.000102,0.0,0.000102,0.0,0.000193,0.000193,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""nfmcg_15740403""",0.017094,0.037736,0.0,0.0,0.0,0.0,0.017094,0.018868,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.5,nu

In [102]:
pl.read_parquet("datamart/features/events/user_id-item_id/01250.pq").sample(10)

user_id,item_id,f_num__user_id_item_id__num_view_30d,f_num__user_id_item_id__num_view_from_u2i_30d,f_num__user_id_item_id__num_view_from_i2i_30d,f_num__user_id_item_id__num_view_from_catalog_30d,f_num__user_id_item_id__num_view_from_search_30d,f_num__user_id_item_id__num_view_from_other_30d,f_num__user_id_item_id__num_click_30d,f_num__user_id_item_id__num_click_from_u2i_30d,f_num__user_id_item_id__num_click_from_i2i_30d,f_num__user_id_item_id__num_click_from_catalog_30d,f_num__user_id_item_id__num_click_from_search_30d,f_num__user_id_item_id__num_click_from_other_30d,f_num__user_id_item_id__num_clickout_30d,f_num__user_id_item_id__num_clickout_from_u2i_30d,f_num__user_id_item_id__num_clickout_from_i2i_30d,f_num__user_id_item_id__num_clickout_from_catalog_30d,f_num__user_id_item_id__num_clickout_from_search_30d,f_num__user_id_item_id__num_clickout_from_other_30d,f_num__user_id_item_id__num_like_30d,f_num__user_id_item_id__num_like_from_u2i_30d,f_num__user_id_item_id__num_like_from_i2i_30d,f_num__user_id_item_id__num_like_from_catalog_30d,f_num__user_id_item_id__num_like_from_search_30d,f_num__user_id_item_id__num_like_from_other_30d,f_num__user_id_item_id__median_price_view_30d,f_num__user_id_item_id__median_price_view_from_u2i_30d,f_num__user_id_item_id__median_price_view_from_i2i_30d,f_num__user_id_item_id__median_price_view_from_catalog_30d,f_num__user_id_item_id__median_price_view_from_search_30d,f_num__user_id_item_id__median_price_view_from_other_30d,f_num__user_id_item_id__median_price_click_30d,f_num__user_id_item_id__median_price_click_from_u2i_30d,f_num__user_id_item_id__median_price_click_from_i2i_30d,f_num__user_id_item_id__median_price_click_from_catalog_30d,f_num__user_id_item_id__median_price_click_from_search_30d,f_num__user_id_item_id__median_price_click_from_other_30d,f_num__user_id_item_id__median_price_clickout_30d,f_num__user_id_item_id__median_price_clickout_from_u2i_30d,f_num__user_id_item_id__median_price_clickout_from_i2i_30d,f_num__user_id_item_id__median_price_clickout_from_catalog_30d,f_num__user_id_item_id__median_price_clickout_from_search_30d,f_num__user_id_item_id__median_price_clickout_from_other_30d,f_num__user_id_item_id__median_price_like_30d,f_num__user_id_item_id__median_price_like_from_u2i_30d,f_num__user_id_item_id__median_price_like_from_i2i_30d,f_num__user_id_item_id__median_price_like_from_catalog_30d,f_num__user_id_item_id__median_price_like_from_search_30d,f_num__user_id_item_id__median_price_like_from_other_30d
u64,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
40415392,"""nfmcg_14930732""",1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
50526312,"""nfmcg_19429126""",1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
82893632,"""nfmcg_20563068""",2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
38373768,"""nfmcg_10502486""",1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.369214,4.369214,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
80616869,"""nfmcg_7237758""",1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
58266396,"""nfmcg_9349775""",1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.

In [103]:
pl.read_parquet("datamart/features/events/socdem_cluster-brand_id/01250.pq").sample(10)

socdem_cluster,brand_id,f_num__socdem_cluster_brand_id__conv_from_view_to_click_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_click_from_u2i_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_click_from_i2i_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_click_from_catalog_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_click_from_search_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_click_from_other_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_clickout_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_clickout_from_u2i_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_clickout_from_i2i_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_clickout_from_catalog_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_clickout_from_search_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_clickout_from_other_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_like_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_like_from_u2i_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_like_from_i2i_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_like_from_catalog_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_like_from_search_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_like_from_other_30d,f_num__socdem_cluster_brand_id__conv_from_click_to_clickout_30d,f_num__socdem_cluster_brand_id__conv_from_click_to_clickout_from_u2i_30d,f_num__socdem_cluster_brand_id__conv_from_click_to_clickout_from_i2i_30d,f_num__socdem_cluster_brand_id__conv_from_click_to_clickout_from_catalog_30d,f_num__socdem_cluster_brand_id__conv_from_click_to_clickout_from_search_30d,f_num__socdem_cluster_brand_id__conv_from_click_to_clickout_from_other_30d,f_num__socdem_cluster_brand_id__conv_from_click_to_like_30d,f_num__socdem_cluster_brand_id__conv_from_click_to_like_from_u2i_30d,f_num__socdem_cluster_brand_id__conv_from_click_to_like_from_i2i_30d,f_num__socdem_cluster_brand_id__conv_from_click_to_like_from_catalog_30d,f_num__socdem_cluster_brand_id__conv_from_click_to_like_from_search_30d,f_num__socdem_cluster_brand_id__conv_from_click_to_like_from_other_30d,f_num__socdem_cluster_brand_id__conv_from_clickout_to_like_30d,f_num__socdem_cluster_brand_id__conv_from_clickout_to_like_from_u2i_30d,f_num__socdem_cluster_brand_id__conv_from_clickout_to_like_from_i2i_30d,f_num__socdem_cluster_brand_id__conv_from_clickout_to_like_from_catalog_30d,f_num__socdem_cluster_brand_id__conv_from_clickout_to_like_from_search_30d,…,f_num__socdem_cluster_brand_id__share_click_from_other_30d,f_num__socdem_cluster_brand_id__share_clickout_30d,f_num__socdem_cluster_brand_id__share_clickout_from_u2i_30d,f_num__socdem_cluster_brand_id__share_clickout_from_i2i_30d,f_num__socdem_cluster_brand_id__share_clickout_from_catalog_30d,f_num__socdem_cluster_brand_id__share_clickout_from_search_30d,f_num__socdem_cluster_brand_id__share_clickout_from_other_30d,f_num__socdem_cluster_brand_id__share_like_30d,f_num__socdem_cluster_brand_id__share_like_from_u2i_30d,f_num__socdem_cluster_brand_id__share_like_from_i2i_30d,f_num__socdem_cluster_brand_id__share_like_from_catalog_30d,f_num__socdem_cluster_brand_id__share_like_from_search_30d,f_num__socdem_cluster_brand_id__share_like_from_other_30d,f_num__socdem_cluster_brand_id__median_price_view_30d,f_num__socdem_cluster_brand_id__median_price_view_from_u2i_30d,f_num__socdem_cluster_brand_id__median_price_view_from_i2i_30d,f_num__socdem_cluster_brand_id__median_price_view_from_catalog_30d,f_num__socdem_cluster_brand_id__median_price_view_from_search_30d,f_num__socdem_cluster_brand_id__median_price_view_from_other_30d,f_num__socdem_cluster_brand_id__median_price_click_30d,f_num__socdem_cluster_brand_id__median_price_click_from_u2i_30d,f_num__socdem_cluster_brand_id__median_price_click_from_i2i_30d,f_num__socdem_cluster_brand_id__median_price_click_from_catalog_30d,f_num__socdem_cluster_brand_id__median_price_click_from_search_30d,f_num__socdem_cluster_brand_id__median_price_click_from_oth

## II. Сбор датасета. (3 балла)

Будем учить модель ранжировать показанные пользователю рекомендации в рамках дня (можно было бы выбрать и другой промежуток). Разделим подготовку обучающих данных на два этапа: сбор "скелета" (базиса) с последующим созданием датасета путем джойна фичей на базис.

### II.I Сбор датасета -> Сбор базиса (1 из 3 баллов)

Базис представляется как (`session_id`, `user_id`, `item_id`, `label`), где в качестве `session_id` используется конкатенация `user_id` и `day`, а `label` зависит от `action_type`. Напомню, что view < click < clickout < like. Сессии, целиком состоящие из view, не стоит учитывать (действительно, сложно оценить качество сортировки одинаковых элементов). Если в рамках сессии было несколько взаимодействий с айтемом, то в качестве `label` нужно взять максимальное значение.

После того, как получим предсказания модели, можно будет сгруппировать базис по `session_id` и получить структуру ([`item_id`], [`label`], [`score`]) - тогда, отсортировав по айтемы по `label` либо `score` получим список айтемов с реальной либо модельной сортировкой, а по этому уже удобно считать метрики.

Реализуйте сбор базиса. Его так же удобно сохранять "за каждый день". Добавьте логику фильтрации сессий по 99 перцентилю длины. 

In [ ]:
def build_basis(
    day_from: int,
    day_to: int,
    filter_99: bool = True,
    output_dir: Path = Path("output/basis/"),
) -> None:
    raw_dir = Path("datamart/raw/events/")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    ## YOUR CODE HERE

In [114]:
build_basis(day_from=min(DAYS), day_to=max(DAYS), filter_99=True)

  0%|          | 0/51 [00:00<?, ?it/s]

In [115]:
basis = pl.read_parquet("output/basis/")
print(basis.shape)
basis.sample(10)

(1815124, 4)


session_id,user_id,item_id,label
str,u64,str,u8
"""1244__5421050""",5421050,"""nfmcg_15608159""",0
"""1297__5755208""",5755208,"""nfmcg_21311703""",0
"""1300__33286522""",33286522,"""nfmcg_4877578""",0
"""1226__53252493""",53252493,"""nfmcg_8102813""",1
"""1299__78840009""",78840009,"""nfmcg_10733492""",0
"""1226__20977""",20977,"""nfmcg_15519510""",0
"""1241__78444691""",78444691,"""nfmcg_9728251""",0
"""1203__69932256""",69932256,"""nfmcg_5600136""",0
"""1294__77643501""",77643501,"""nfmcg_23642999""",0


In [116]:
for col in basis.columns:
    print(f"{col}: {basis[col].n_unique()}")

session_id: 81998
user_id: 38562
item_id: 24531
label: 4


In [117]:
basis["label"].value_counts().sort("label")

label,count
u8,u32
0,1638347
1,148402
2,25704
3,2671


In [118]:
basis.group_by("session_id").agg(pl.len())["len"].describe()

statistic,value
str,f64
"""count""",81998.0
"""null_count""",0.0
"""mean""",22.136198
"""std""",21.801584
"""min""",2.0
"""25%""",7.0
"""50%""",15.0
"""75%""",31.0
"""max""",152.0


### II.II Сбор датасета -> Сбор датасета с фичами (2 из 3 баллов)

Чтобы создать датасет, достаточно приджойнить к базису фичи. Обратите внимание, что фичи должны быть собраны за предыдущий день, чтобы избежать ликов. То есть, если мы работаем с базисом на 1300 день, то фичи для него необходимо брать из 1299 дня. Помимо числовых, можно также добавить категориальные фичи.

Реализуйте необходимую логику. Добавьте возможность читать список фичей, которые необходимо приджойнить, из файла. 

In [ ]:
def build_dataset(
    day_from: int,
    day_to: int,
    basis_dir: Path = Path("output/basis/"),
    output_dir: Path = Path("output/dataset/"),
    features_to_use_filepath: Path | None = None,
) -> None:

    ## YOUR CODE HERE

In [ ]:
def join_features(
    df: pl.LazyFrame,
    day: int,
    users: pl.LazyFrame,
    items: pl.LazyFrame,
    features_dir: Path = Path("datamart/features/events/"),
    features_to_use: list[str] | None = None,
) -> pl.LazyFrame:

    ## YOUR CODE HERE


def build_dataset(
    day_from: int,
    day_to: int,
    basis_dir: Path = Path("output/basis/"),
    output_dir: Path = Path("output/dataset/"),
    features_to_use_filepath: Path | None = None,
    users: pl.LazyFrame | None = None,
    items: pl.LazyFrame | None = None,
) -> None:

    ## YOUR CODE HERE

In [121]:
build_dataset(day_from=min(DAYS), day_to=max(DAYS), output_dir = Path("output/dataset/"))

  0%|          | 0/51 [00:00<?, ?it/s]

In [122]:
pl.read_parquet("output/dataset/01250.pq").sample(10)

session_id,user_id,item_id,label,f_cat__socdem_cluster,f_cat__region,f_cat__brand_id,f_cat__category,f_cat__subcategory,f_num__price,embedding,f_num__socdem_cluster_category__conv_from_view_to_click_30d,f_num__socdem_cluster_category__conv_from_view_to_click_from_u2i_30d,f_num__socdem_cluster_category__conv_from_view_to_click_from_i2i_30d,f_num__socdem_cluster_category__conv_from_view_to_click_from_catalog_30d,f_num__socdem_cluster_category__conv_from_view_to_click_from_search_30d,f_num__socdem_cluster_category__conv_from_view_to_click_from_other_30d,f_num__socdem_cluster_category__conv_from_view_to_clickout_30d,f_num__socdem_cluster_category__conv_from_view_to_clickout_from_u2i_30d,f_num__socdem_cluster_category__conv_from_view_to_clickout_from_i2i_30d,f_num__socdem_cluster_category__conv_from_view_to_clickout_from_catalog_30d,f_num__socdem_cluster_category__conv_from_view_to_clickout_from_search_30d,f_num__socdem_cluster_category__conv_from_view_to_clickout_from_other_30d,f_num__socdem_cluster_category__conv_from_view_to_like_30d,f_num__socdem_cluster_category__conv_from_view_to_like_from_u2i_30d,f_num__socdem_cluster_category__conv_from_view_to_like_from_i2i_30d,f_num__socdem_cluster_category__conv_from_view_to_like_from_catalog_30d,f_num__socdem_cluster_category__conv_from_view_to_like_from_search_30d,f_num__socdem_cluster_category__conv_from_view_to_like_from_other_30d,f_num__socdem_cluster_category__conv_from_click_to_clickout_30d,f_num__socdem_cluster_category__conv_from_click_to_clickout_from_u2i_30d,f_num__socdem_cluster_category__conv_from_click_to_clickout_from_i2i_30d,f_num__socdem_cluster_category__conv_from_click_to_clickout_from_catalog_30d,f_num__socdem_cluster_category__conv_from_click_to_clickout_from_search_30d,f_num__socdem_cluster_category__conv_from_click_to_clickout_from_other_30d,f_num__socdem_cluster_category__conv_from_click_to_like_30d,f_num__socdem_cluster_category__conv_from_click_to_like_from_u2i_30d,…,f_num__socdem_cluster_brand_id__share_click_from_other_30d,f_num__socdem_cluster_brand_id__share_clickout_30d,f_num__socdem_cluster_brand_id__share_clickout_from_u2i_30d,f_num__socdem_cluster_brand_id__share_clickout_from_i2i_30d,f_num__socdem_cluster_brand_id__share_clickout_from_catalog_30d,f_num__socdem_cluster_brand_id__share_clickout_from_search_30d,f_num__socdem_cluster_brand_id__share_clickout_from_other_30d,f_num__socdem_cluster_brand_id__share_like_30d,f_num__socdem_cluster_brand_id__share_like_from_u2i_30d,f_num__socdem_cluster_brand_id__share_like_from_i2i_30d,f_num__socdem_cluster_brand_id__share_like_from_catalog_30d,f_num__socdem_cluster_brand_id__share_like_from_search_30d,f_num__socdem_cluster_brand_id__share_like_from_other_30d,f_num__socdem_cluster_brand_id__median_price_view_30d,f_num__socdem_cluster_brand_id__median_price_view_from_u2i_30d,f_num__socdem_cluster_brand_id__median_price_view_from_i2i_30d,f_num__socdem_cluster_brand_id__median_price_view_from_catalog_30d,f_num__socdem_cluster_brand_id__median_price_view_from_search_30d,f_num__socdem_cluster_brand_id__median_price_view_from_other_30d,f_num__socdem_cluster_brand_id__median_price_click_30d,f_num__socdem_cluster_brand_id__median_price_click_from_u2i_30d,f_num__socdem_cluster_brand_id__median_price_click_from_i2i_30d,f_num__socdem_cluster_brand_id__median_price_click_from_catalog_30d,f_num__socdem_cluster_brand_id__median_price_click_from_search_30d,f_num__socdem_cluster_brand_id__median_price_click_from_other_30d,f_num__socdem_cluster_brand_id__median_price_clickout_30d,f_num__socdem_cluster_brand_id__median_price_clickout_from_u2i_30d,f_num__socdem_cluster_brand_id__median_price_clickout_from_i2i_30d,f_num__socdem_cluster_brand_id__median_price_clickout_from_catalog_30d,f_num__socdem_cluster_brand_id__median_price_clickout_from_search_30d,f_num__socdem_cluster_brand_id__median_price_clickout_from_other_30d,f_num__socdem_cluster_brand_id__median_price_like_30d,f_num__socdem_cluster_brand_id__median_price_like_from_u2i_30

## III Обучение ранжирования (4 балла)

### III.I. Подготовка выборок для обучения/валидации/теста и реализация метрик (1 из 4 баллов)

Полезно будет также написать функцию, считывающую с диска и возвращающую train, val, train_val, и test части датасета. Диапазон будем задавать через дни.

In [ ]:
def read_dataset(
    day_from: int,
    n_train_days: int,
    n_val_days: int,
    n_test_days: int,
    dataset_dir: Path = Path("output/dataset/"),
) -> tuple[pl.LazyFrame, pl.LazyFrame, pl.LazyFrame, pl.LazyFrame]:
    ## YOUR CODE HERE

In [124]:
train_df, val_df, train_val_df, test_df = read_dataset(
    day_from=1265,
    n_train_days=14,
    n_val_days=1,
    n_test_days=1,
    dataset_dir=Path("output/dataset/")
)
train_df = train_df.collect()
val_df = val_df.collect()
train_val_df = train_val_df.collect()
test_df = test_df.collect()

/var/folders/gw/9vsxt6xx1t39l8d2vg28flr00000gq/T/ipykernel_57004/1207791258.py:8: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  train_df = train_df.collect()
/var/folders/gw/9vsxt6xx1t39l8d2vg28flr00000gq/T/ipykernel_57004/1207791258.py:10: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  train_val_df = train_val_df.collect()


In [125]:
train_val_df.shape

(99686, 635)

In [126]:
test_df.shape

(7128, 635)

Реализуйте логику расчета метрик по структуре ([`item_id`], [`label`], [`score`]) - можете формировать эту структуру также внутри функции расчета метрик, а можете вне. В качестве метрики обязательно использовать NDCG@k. В выборе остальных метрик вы свободны. Полезно может быть считать метрики в разрезе по label - например, сколько айтемов с label=2 попали в топ-10 рекомендаций.

Посчитайте метрики в A/A-сеттинге.

In [ ]:
class AtKMetric(ABC):
    def __init__(self, k: int):
        self.k = k

    @property
    @abstractmethod
    def name(self) -> str:
        raise NotImplementedError

    @property
    def full_name(self) -> str:
        return f"{self.name}@{self.k}"

    @abstractmethod
    def __call__(self, *, labels_col: str = "labels", targets_col: str = "targets") -> pl.Expr:
        raise NotImplementedError


class NdcgAtK(AtKMetric):
    @property
    def name(self) -> str:
        return "ndcg"
    ## YOUR CODE HERE


def evaluate_ranker(
    df: pl.DataFrame,
    ks: list[int] = [1,  5, 10, 20, 50],
    preds_col: str = "preds",
    targets_col: str = "targets",
) -> pl.DataFrame:
    ## YOUR CODE HERE

In [136]:
metrics_best = evaluate_ranker(
    test_df.with_columns(score=pl.col("label"))
    .group_by("session_id").agg(
        [
            pl.struct("item_id").sort_by("score", descending=True).alias("preds"),
            pl.struct("item_id", "label").sort_by("label", descending=True).alias("targets"),
        ]
    )
)
metrics_worst = evaluate_ranker(
    test_df.with_columns(score=pl.col("label"))
    .group_by("session_id").agg(
        [
            pl.struct("item_id").sort_by("score", descending=False).alias("preds"),
            pl.struct("item_id", "label").sort_by("label", descending=True).alias("targets"),
        ]
    )
)
RESULTS = pd.concat([
    pd.DataFrame(metrics_best, index=["best"]),
    pd.DataFrame(metrics_worst, index=["worst"]), 
])
RESULTS.style.format(precision=5).background_gradient(cmap="Blues")

,ndcg@1,ndcg@5,ndcg@10,ndcg@20,ndcg@50
best,1.00000,1.00000,1.00000,1.00000,1.00000
worst,0.00980,0.11265,0.22007,0.29893,0.35665


### III.II. Реализация TopPopular бейзлайна (1 из 4 баллов)

Реализуйте любой бейзлайн (бейзлайны) на своё усмотрение. Посчитайте метрики. Не забудьте про консистентность: учимся на train - оцениваем на val; учимся на train+val - оцениваем на test.

Важно: используйте `sample(fraction=1.0, shuffle=True)` при группировке по сессии для расчета метрик, чтобы в случае одинаковых скоров автоматом не проставлялся скор из корркетно отсортированной последовтаельности айтемов! 

In [ ]:
baseline = evaluate_ranker(
    ## YOUR CODE HERE
)

RESULTS = pd.concat([
    RESULTS,
    pd.DataFrame(baseline, index=["baseline"]),
])
RESULTS.style.format(precision=5).background_gradient(cmap="Blues")

,ndcg@1,ndcg@5,ndcg@10,ndcg@20,ndcg@50
best,1.00000,1.00000,1.00000,1.00000,1.00000
worst,0.00980,0.11265,0.22007,0.29893,0.35665
baseline,0.17211,0.35192,0.44606,0.49917,0.52305


In [138]:
test_df['user_id'].n_unique()

459

### III.III. Реализация обучения градиентного бустинга (2 из 4 баллов)

Обучите ранкер на train части датасета. В качестве модели можете использовать любую из [Catboost](https://catboost.ai), [LGBM](https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.Booster.html), [XGBoost](https://xgboost.readthedocs.io/en/stable/). Обучать можно как Ranker, так и Classifier, так и Regressor. Поэкспериментируйте. Посчитайте метрики на val части и подберите гиперпараметры (можете взять разные временные срезы, чтобы не заоверфиттиться под один).

Обучите итоговую модель на train + val, замерьте качество на test и сравните с бейзлайном.

Sanity check. Обратите внимание, что если вы считаете бейзлайн по фиче из датасета, то фича, по которой вы считаете бейзлайн, обязательно должна присутствовать как фича для ранкера. Если ранкер при использовании этой фичи показывает результаты хуже, чем бейзлайн, то что-то с вашим ранкером не так.

In [139]:
features = [col for col in train_df.columns if col.startswith("f_")]
categorical_features = [col for col in features if col.startswith("f_cat__")]

In [ ]:
def train_evaluate_model(params, train_df, val_df, features, categorical_features):

    ## YOUR CODE HERE


study = ...
print(f"\nBest ndcg@5: {study.best_value:.5f}")

[I 2026-04-03 14:22:03,712] A new study created in memory with name: no-name-8e6425fe-e42d-4939-84de-1910275e2410


  0%|          | 0/7 [00:00<?, ?it/s]

[I 2026-04-03 14:22:22,785] Trial 0 finished with value: 0.3314310610294342 and parameters: {'num_leaves': 87, 'max_depth': 29, 'learning_rate': 0.1205712628744377, 'n_estimators': 340, 'reg_alpha': 0.004207988669606638, 'reg_lambda': 0.004207053950287938, 'subsample': 0.5290418060840998, 'colsample_bytree': 0.9330880728874675, 'lambdarank_truncation_level': 64}. Best is trial 0 with value: 0.3314310610294342.
[I 2026-04-03 14:22:29,085] Trial 1 finished with value: 0.35038554668426514 and parameters: {'num_leaves': 148, 'max_depth': 3, 'learning_rate': 0.27081608642499677, 'n_estimators': 433, 'reg_alpha': 0.0070689749506246055, 'reg_lambda': 0.005337032762603957, 'subsample': 0.5917022549267169, 'colsample_bytree': 0.6521211214797689, 'lambdarank_truncation_level': 57}. Best is trial 1 with value: 0.35038554668426514.
[I 2026-04-03 14:22:38,592] Trial 2 finished with value: 0.32284384965896606 and parameters: {'num_leaves': 98, 'max_depth': 11, 'learning_rate': 0.08012737503998542, '

In [ ]:
## YOUR CODE HERE
RESULTS = pd.concat([
    RESULTS,
    pd.DataFrame(ranker, index=["ranker"]),
])
RESULTS.style.format(precision=5).background_gradient(cmap="Blues")

,ndcg@1,ndcg@5,ndcg@10,ndcg@20,ndcg@50
best,1.00000,1.00000,1.00000,1.00000,1.00000
worst,0.00980,0.11265,0.22007,0.29893,0.35665
baseline,0.17211,0.35192,0.44606,0.49917,0.52305
ranker,0.21133,0.37737,0.46745,0.51507,0.54176


Опишите полученный результат - получилось ли обогнать бустингом baseline? Почему?